## Creating fact view

In [0]:
%sql
CREATE OR REPLACE VIEW transport_lakehouse.gold.fact_public_vehicles AS
SELECT
  p.car_num,
  m.manufacturer_key,
  vt.vehicle_type_key,
  c.color_key,
  P.commercial_nm,
  p.total_weight,
  p.capacity_amount,
  p.prod_year,
  p.test_valid_until_dt,
  p.canceled_cd,
  p.canceled_nm,
  p.canceled_dt,
  1 AS vehicle_count

FROM transport_lakehouse.silver.silver_vehicles_public p
LEFT JOIN transport_lakehouse.gold.dim_manufacturer m
  ON CAST(p.manufacturer_cd AS STRING)=CAST(m.manufacturer_cd AS STRING)
LEFT JOIN transport_lakehouse.gold.dim_vehicle_type vt
  ON CAST(p.vehicle_type_cd AS STRING)=CAST(vt.vehicle_type_cd AS STRING)
 AND p.vehicle_type_nm = vt.vehicle_type_nm
LEFT JOIN transport_lakehouse.gold.dim_color c
  ON CAST(p.color_cd AS STRING)=CAST(c.color_cd AS STRING)
ORDER BY p.canceled_cd


## Missing keys check

In [0]:
%sql
SELECT
  SUM(CASE WHEN manufacturer_key IS NULL THEN 1 ELSE 0 END) AS missing_manufacturer_key,
  SUM(CASE WHEN vehicle_type_key IS NULL THEN 1 ELSE 0 END) AS missing_vehicle_type_key,
  SUM(CASE WHEN color_key IS NULL THEN 1 ELSE 0 END) AS missing_color_key
FROM transport_lakehouse.gold.fact_public_vehicles


## Duplication check

In [0]:
%sql
SELECT COUNT(*) AS rows, COUNT(DISTINCT car_num) AS distinct_cars
FROM transport_lakehouse.gold.fact_public_vehicles

## Key null ratio check

In [0]:
%sql
SELECT
  CAST(AVG(CASE WHEN manufacturer_key IS NOT NULL THEN 1 ELSE 0 END) AS FLOAT) AS manufacturer_key_fill_rate,
  CAST(AVG(CASE WHEN vehicle_type_key IS NOT NULL THEN 1 ELSE 0 END) AS FLOAT) AS vehicle_type_key_fill_rate,
  CAST(AVG(CASE WHEN color_key IS NOT NULL THEN 1 ELSE 0 END) AS FLOAT) AS       color_key_fill_rate
FROM transport_lakehouse.gold.fact_public_vehicles
